<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)

import sys
sys.path.append('/content/drive/MyDrive/ml_project')



from choose_source import choose_source
from load_data import load_data
from select_columns import select_columns_ui, select_columns
from detect_types import detect_types
from set_date import set_date_ui, set_date
import clean_data
import importlib
from clean_data import clean_data_ui, clean_data
from date_features import date_features_ui, date_features
from create_lags import create_lags_ui, create_lags
from encode_categories import encode_categories_ui, encode_categories
from rolling_stats import rolling_stats_ui, rolling_stats
from set_target import set_target_ui, set_target
from scale_data import scale_data_ui, scale_data
from split_time import split_time_ui, split_time
from create_traditional_model import create_traditional_model_ui
from create_ml_model import create_ml_model_ui
from train_ml import train_ml
from predict_ml import predict_ml
from train_traditional import train_traditional



Mounted at /content/drive
Mounted at /content/drive


In [ ]:
# -----------------------------------------------------------
# step 1: select and load the data source (ui)
# -----------------------------------------------------------
# this step displays an interactive ui that lets the user choose:
#  - the input data format (csv, json, api)
#  - the input method (file path or file upload)
#  - the actual file path or api url
#
# based on the user's selection:
#  - if csv/json is chosen, the file is copied into:
#        /content/drive/MyDrive/ml_project/upload.csv
#        or
#        /content/drive/MyDrive/ml_project/upload.json
#  - if api is chosen, data is downloaded directly from the url
#
# output:
#  - prints a preview of the loaded dataframe
#  - the loaded file is saved as:
#       upload.csv  or  upload.json
# the dataframe itself is held in memory only at this step
#
# no other files are created here; this step only loads data.
choose_source()


In [ ]:
# -----------------------------------------------------------
# step 2: load the previously selected data source
# -----------------------------------------------------------
# this step detects which input file was created in step 1:
#   - upload.csv       (if the user selected a csv file)
#   - upload.json      (if the user selected a json file)
#   - api_url.txt      (if the user selected an api source)
#
# based on the detected file, the function:
#   1. reads the csv/json file or downloads data from the api url
#   2. converts the data into a pandas dataframe
#   3. saves the dataframe as:
#        df.pkl   (main working dataset for all following modules)
#
# input files:
#   upload.csv or upload.json or api_url.txt
#
# output files:
#   df.pkl   (full raw dataset stored in the project directory)
#
# the loaded dataframe is also returned and previewed in the notebook.
df = load_data()
print(df.head())


DataFrame saved to: /content/drive/MyDrive/ml_project/df.pkl
      PO_ID         Supplier  Order_Date Delivery_Date    Item_Category  \
0  PO-00001        Alpha_Inc  2023-10-17    2023-10-25  Office Supplies   
1  PO-00002  Delta_Logistics  2022-04-25    2022-05-05  Office Supplies   
2  PO-00003         Gamma_Co  2022-01-26    2022-02-15              MRO   
3  PO-00004    Beta_Supplies  2022-10-09    2022-10-28        Packaging   
4  PO-00005  Delta_Logistics  2022-09-08    2022-09-20    Raw Materials   

  Order_Status  Quantity  Unit_Price  Negotiated_Price  Defective_Units  \
0    Cancelled      1176       20.13             17.81              NaN   
1    Delivered      1509       39.32             37.34            235.0   
2    Delivered       910       95.51             92.26             41.0   
3    Delivered      1344       99.85             95.52            112.0   
4    Delivered      1180       64.07             60.53            171.0   

  Compliance  
0        Yes  
1      

In [ ]:
# -----------------------------------------------------------
# step 3: select columns for further processing (ui)
# -----------------------------------------------------------
# this step opens a checkbox-based ui showing all columns from:
#
#   df.pkl
#
# which was created in step 2 by the load_data() function.
#
# the user selects which columns should be kept in the project.
#
# input file:
#   /content/drive/MyDrive/ml_project/df.pkl
#
# output file:
#   /content/drive/MyDrive/ml_project/selected_columns.txt
#       - contains one column name per line
#       - used in step 4 to filter the dataframe
#
# no dataframe is saved in this step — only the list of selected columns.
select_columns_ui(df)

In [ ]:

# -----------------------------------------------------------
# step 4: apply selected columns and create df_selected.pkl
# -----------------------------------------------------------
# this step:
#   1. loads the full dataset:
#        /content/drive/MyDrive/ml_project/df.pkl
#   2. loads the selected column list:
#        /content/drive/MyDrive/ml_project/selected_columns.txt
#   3. filters the dataframe to keep only the chosen columns
#   4. saves the filtered dataframe as:
#        /content/drive/MyDrive/ml_project/df_selected.pkl
#
# input files:
#   df.pkl
#   selected_columns.txt
#
# output file:
#   df_selected.pkl
#
# the resulting dataframe is also returned to the notebook for preview.
df_selected = select_columns()
print(df_selected.head())

saved: /content/drive/MyDrive/ml_project/df_selected.pkl
selected columns: ['PO_ID', 'Supplier', 'Order_Date', 'Delivery_Date', 'Item_Category', 'Order_Status', 'Quantity', 'Unit_Price']
      PO_ID         Supplier  Order_Date Delivery_Date    Item_Category  \
0  PO-00001        Alpha_Inc  2023-10-17    2023-10-25  Office Supplies   
1  PO-00002  Delta_Logistics  2022-04-25    2022-05-05  Office Supplies   
2  PO-00003         Gamma_Co  2022-01-26    2022-02-15              MRO   
3  PO-00004    Beta_Supplies  2022-10-09    2022-10-28        Packaging   
4  PO-00005  Delta_Logistics  2022-09-08    2022-09-20    Raw Materials   

  Order_Status  Quantity  Unit_Price  
0    Cancelled      1176       20.13  
1    Delivered      1509       39.32  
2    Delivered       910       95.51  
3    Delivered      1344       99.85  
4    Delivered      1180       64.07  


In [ ]:
# -----------------------------------------------------------
# step 5: detect column data types in df_selected.pkl
# -----------------------------------------------------------
# this step loads the filtered dataset:
#
#   /content/drive/MyDrive/ml_project/df_selected.pkl
#
# it analyzes each column and classifies it into one of the groups:
#   - numerical   (int64, float64)
#   - categorical (object, category)
#   - datetime    (datetime64[ns])
#
# the function then saves a detailed table listing every column
# and its detected type into:
#
#   /content/drive/MyDrive/ml_project/column_types.csv
#
# input file:
#   df_selected.pkl
#
# output file:
#   column_types.csv
#
# the function also returns a dictionary containing three lists:
# { "numerical": [...], "categorical": [...], "datetime": [...] }
types_dict = detect_types()




saved: /content/drive/MyDrive/ml_project/column_types.csv
{'numerical': ['Quantity', 'Unit_Price'], 'categorical': ['PO_ID', 'Supplier', 'Order_Date', 'Delivery_Date', 'Item_Category', 'Order_Status'], 'datetime': []}


In [ ]:
# -----------------------------------------------------------
# step 6: select the date column and the fill method (ui)
# -----------------------------------------------------------
# this step loads:
#
#   /content/drive/MyDrive/ml_project/df_selected.pkl
#
# and displays a ui that allows the user to select:
#   1. which column should be interpreted as the date column
#   2. how missing date values should be handled:
#        - zero   → fill with timestamp(0)
#        - ffill  → forward fill
#        - mean   → fill with the mean timestamp
#
# the user’s selections are saved into:
#
#   /content/drive/MyDrive/ml_project/date_settings.txt
#
# (line 1 = selected date column, line 2 = fill method)
#
# this step does not modify the dataframe; it only stores the settings.
set_date_ui()


In [ ]:
# -----------------------------------------------------------
# step 7: apply date settings and create df_date.pkl
# -----------------------------------------------------------
# this step:
#   1. loads the filtered dataset:
#        /content/drive/MyDrive/ml_project/df_selected.pkl
#   2. loads the user’s date configuration:
#        /content/drive/MyDrive/ml_project/date_settings.txt
#   3. converts the selected column to datetime
#   4. sorts the dataset chronologically by that column
#   5. fills missing values using the method stored in date_settings.txt:
#        - zero   → fill with timestamp(0)
#        - ffill  → forward fill
#        - mean   → fill with mean timestamp
#   6. saves the processed dataframe as:
#        /content/drive/MyDrive/ml_project/df_date.pkl
#
# input files:
#   df_selected.pkl
#   date_settings.txt
#
# output file:
#   df_date.pkl
#
# the resulting dataframe is returned for preview.
df_date = set_date()
df_date.head()


saved: /content/drive/MyDrive/ml_project/df_date.pkl


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price
558,PO-00559,Alpha_Inc,2022-01-01,2022-01-07,Raw Materials,Delivered,1042,36.74
179,PO-00180,Alpha_Inc,2022-01-01,2022-01-12,Electronics,Delivered,552,21.52
521,PO-00522,Delta_Logistics,2022-01-02,2022-01-06,Packaging,Delivered,85,22.54
302,PO-00303,Alpha_Inc,2022-01-03,2022-01-20,Packaging,Delivered,172,63.61
420,PO-00421,Alpha_Inc,2022-01-04,2022-01-06,Electronics,Delivered,1004,50.70


In [ ]:
# -----------------------------------------------------------
# step 8: inspect missing values and choose cleaning rules (ui)
# -----------------------------------------------------------
# this step loads the dataset:
#
#   /content/drive/MyDrive/ml_project/df_date.pkl
#
# for every column, the ui displays:
#   - the column name
#   - the number of missing values
#   - a dropdown allowing the user to choose a cleaning rule:
#       drop  → remove all rows with missing values in this column
#       zero  → fill missing values with 0
#       ffill → forward fill missing values
#       mean  → fill missing numeric values with the column mean
#       none  → leave missing values unchanged
#
# the selected rule for each column is saved line-by-line into:
#
#   /content/drive/MyDrive/ml_project/clean_method.txt
#
# input file:
#   df_date.pkl
#
# output file:
#   clean_method.txt  (per-column cleaning configuration)
clean_data_ui()



In [ ]:
# -----------------------------------------------------------
# step 9: apply cleaning rules and create df_clean.pkl
# -----------------------------------------------------------
# this step:
#   1. loads the dataset:
#        /content/drive/MyDrive/ml_project/df_date.pkl
#   2. loads the cleaning configuration:
#        /content/drive/MyDrive/ml_project/clean_method.txt
#      (each line formatted as: column_name:method)
#   3. applies the selected method per column:
#        drop  → remove all rows with missing values in that column
#        zero  → fill missing values with 0
#        ffill → forward fill missing values
#        mean  → fill with column mean (numeric only)
#        none  → no action
#   4. saves the cleaned dataset as:
#        /content/drive/MyDrive/ml_project/df_clean.pkl
#
# input files:
#   df_date.pkl
#   clean_method.txt
#
# output file:
#   df_clean.pkl
#
# the cleaned dataframe is returned and previewed.
df_clean = clean_data()
df_clean.head()


saved: /content/drive/MyDrive/ml_project/df_clean.pkl
applied per-column cleaning methods


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price
558,PO-00559,Alpha_Inc,2022-01-01,2022-01-07,Raw Materials,Delivered,1042,36.74
179,PO-00180,Alpha_Inc,2022-01-01,2022-01-12,Electronics,Delivered,552,21.52
521,PO-00522,Delta_Logistics,2022-01-02,2022-01-06,Packaging,Delivered,85,22.54
302,PO-00303,Alpha_Inc,2022-01-03,2022-01-20,Packaging,Delivered,172,63.61
420,PO-00421,Alpha_Inc,2022-01-04,2022-01-06,Electronics,Delivered,1004,50.70


In [ ]:
# -----------------------------------------------------------
# step 10: select the date column for calendar feature creation (ui)
# -----------------------------------------------------------
# this step loads the cleaned dataset:
#
#   /content/drive/MyDrive/ml_project/df_clean.pkl
#
# the ui displays a dropdown listing all columns in the dataframe.
# the user selects the column that should be used to generate
# calendar-based features (year, month, day, dayofweek).
#
# the selected column name is saved into:
#
#   /content/drive/MyDrive/ml_project/date_features_col.txt
#
# input file:
#   df_clean.pkl
#
# output file:
#   date_features_col.txt  (selected date column)
date_features_ui()


In [ ]:
# -----------------------------------------------------------
# step 11: generate calendar features and create df_date_features.pkl
# -----------------------------------------------------------
# this step:
#   1. loads the cleaned dataset:
#        /content/drive/MyDrive/ml_project/df_clean.pkl
#   2. loads the selected date column from:
#        /content/drive/MyDrive/ml_project/date_features_col.txt
#   3. converts the chosen column to datetime
#   4. generates four calendar features:
#        year       → df["year"]
#        month      → df["month"]
#        day        → df["day"]
#        dayofweek  → df["dayofweek"]
#   5. saves the result as:
#        /content/drive/MyDrive/ml_project/df_date_features.pkl
#
# input files:
#   df_clean.pkl
#   date_features_col.txt
#
# output file:
#   df_date_features.pkl
#
# the dataframe with new calendar features is returned for preview.
df_date_features = date_features()
df_date_features.head()


saved: /content/drive/MyDrive/ml_project/df_date_features.pkl


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,year,month,day,dayofweek
558,PO-00559,Alpha_Inc,2022-01-01,2022-01-07,Raw Materials,Delivered,1042,36.74,2022,1,1,5
179,PO-00180,Alpha_Inc,2022-01-01,2022-01-12,Electronics,Delivered,552,21.52,2022,1,1,5
521,PO-00522,Delta_Logistics,2022-01-02,2022-01-06,Packaging,Delivered,85,22.54,2022,1,2,6
302,PO-00303,Alpha_Inc,2022-01-03,2022-01-20,Packaging,Delivered,172,63.61,2022,1,3,0
420,PO-00421,Alpha_Inc,2022-01-04,2022-01-06,Electronics,Delivered,1004,50.70,2022,1,4,1


In [ ]:
# -----------------------------------------------------------
# step 12: select the column and lag values for lag feature generation (ui)
# -----------------------------------------------------------
# this step loads the dataset containing calendar features:
#
#   /content/drive/MyDrive/ml_project/df_date_features.pkl
#
# the ui displays:
#   • a dropdown listing all dataframe columns
#   • a text box where the user enters lag values, e.g.: 1,2,7,14
#
# the user selects:
#   1. the column for which lag features should be generated
#   2. a comma-separated list of lag offsets
#
# both selections are saved into:
#
#   /content/drive/MyDrive/ml_project/lags_method.txt
#
# file format:
#   line 1 = selected column
#   line 2 = raw lag string (e.g. "1,2,7")
#
# input file:
#   df_date_features.pkl
#
# output file:
#   lags_method.txt
create_lags_ui()


In [ ]:
# -----------------------------------------------------------
# step 13: generate lag features based on user configuration
# -----------------------------------------------------------
# this step:
#   1. loads the dataset:
#        /content/drive/MyDrive/ml_project/df_date_features.pkl
#   2. loads the lag configuration:
#        /content/drive/MyDrive/ml_project/lags_method.txt
#      (line 1 = column name, line 2 = comma-separated lag values)
#   3. parses each lag value into an integer
#   4. for every lag value, creates a new shifted column:
#        <column>_lag_<lag>  =  df[column].shift(lag)
#      for example:
#        sales_lag_1, sales_lag_2, sales_lag_7
#   5. saves the final dataframe with lag features as:
#        /content/drive/MyDrive/ml_project/df_lags.pkl
#
# input files:
#   df_date_features.pkl
#   lags_method.txt
#
# output file:
#   df_lags.pkl
#
# the dataframe with all generated lag features is returned for preview.
create_lags()


saved: /content/drive/MyDrive/ml_project/df_lags.pkl
generated lag columns


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,year,month,day,dayofweek,Order_Date_lag_5
558,PO-00559,Alpha_Inc,2022-01-01,2022-01-07,Raw Materials,Delivered,1042,36.74,2022,1,1,5,NaT
179,PO-00180,Alpha_Inc,2022-01-01,2022-01-12,Electronics,Delivered,552,21.52,2022,1,1,5,NaT
521,PO-00522,Delta_Logistics,2022-01-02,2022-01-06,Packaging,Delivered,85,22.54,2022,1,2,6,NaT
302,PO-00303,Alpha_Inc,2022-01-03,2022-01-20,Packaging,Delivered,172,63.61,2022,1,3,0,NaT
420,PO-00421,Alpha_Inc,2022-01-04,2022-01-06,Electronics,Delivered,1004,50.70,2022,1,4,1,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
663,PO-00664,Alpha_Inc,2023-12-25,2024-01-12,Electronics,Delivered,1732,15.37,2023,12,25,0,2023-12-20
232,PO-00233,Epsilon_Group,2023-12-26,2024-01-04,MRO,Delivered,1546,43.47,2023,12,26,1,2023-12-20
519,PO-00520,Delta_Logistics,2023-12-27,2023-12-28,Raw Materials,Delivered,952,74.72,2023,12,27,2,2023-12-22
210,PO-00211,Gamma_Co,2024-01-01,NaN,Office Supplies,Delivered,1233,86.62,2024,1,1,0,2023-12-23


In [ ]:
# -----------------------------------------------------------
# step 14: select column, window size, and aggregation method for rolling statistics (ui)
# -----------------------------------------------------------
# this step loads the dataset containing lag features:
#
#   /content/drive/MyDrive/ml_project/df_lags.pkl
#
# the ui displays:
#   • a dropdown with all dataframe columns
#   • a text field to enter the rolling window size (e.g. 3, 7, 14)
#   • a dropdown selecting the aggregation method:
#         mean, sum, median
#
# the user selections are saved into:
#
#   /content/drive/MyDrive/ml_project/rolling_method.txt
#
# file format:
#   line 1 = selected column
#   line 2 = window size (string)
#   line 3 = aggregation method (mean/sum/median)
#
# input file:
#   df_lags.pkl
#
# output file:
#   rolling_method.txt
rolling_stats_ui()



In [ ]:
# -----------------------------------------------------------
# step 15: generate rolling statistics feature and create df_rolling.pkl
# -----------------------------------------------------------
# this step:
#   1. loads the dataset with lag features:
#        /content/drive/MyDrive/ml_project/df_lags.pkl
#   2. loads rolling configuration:
#        /content/drive/MyDrive/ml_project/rolling_method.txt
#      (column, window size, aggregation method)
#   3. converts the window size to an integer
#   4. computes the rolling feature using:
#        dataframe[column].rolling(window).<aggregation>()
#      examples:
#        sales_mean_7
#        demand_sum_14
#        profit_median_3
#   5. saves the resulting dataframe as:
#        /content/drive/MyDrive/ml_project/df_rolling.pkl
#
# input files:
#   df_lags.pkl
#   rolling_method.txt
#
# output file:
#   df_rolling.pkl
#
# the updated dataframe is returned for preview.
rolling_stats()



saved: /content/drive/MyDrive/ml_project/df_rolling.pkl
generated rolling feature: Quantity_mean_5


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,year,month,day,dayofweek,Order_Date_lag_5,Quantity_mean_5
558,PO-00559,Alpha_Inc,2022-01-01,2022-01-07,Raw Materials,Delivered,1042,36.74,2022,1,1,5,NaT,NaN
179,PO-00180,Alpha_Inc,2022-01-01,2022-01-12,Electronics,Delivered,552,21.52,2022,1,1,5,NaT,NaN
521,PO-00522,Delta_Logistics,2022-01-02,2022-01-06,Packaging,Delivered,85,22.54,2022,1,2,6,NaT,NaN
302,PO-00303,Alpha_Inc,2022-01-03,2022-01-20,Packaging,Delivered,172,63.61,2022,1,3,0,NaT,NaN
420,PO-00421,Alpha_Inc,2022-01-04,2022-01-06,Electronics,Delivered,1004,50.70,2022,1,4,1,NaT,571.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663,PO-00664,Alpha_Inc,2023-12-25,2024-01-12,Electronics,Delivered,1732,15.37,2023,12,25,0,2023-12-20,711.2
232,PO-00233,Epsilon_Group,2023-12-26,2024-01-04,MRO,Delivered,1546,43.47,2023,12,26,1,2023-12-20,786.0
519,PO-00520,Delta_Logistics,2023-12-27,2023-12-28,Raw Materials,Delivered,952,74.72,2023,12,27,2,2023-12-22,915.8
210,PO-00211,Gamma_Co,2024-01-01,NaN,Office Supplies,Delivered,1233,86.62,2024,1,1,0,2023-12-23,1148.4


In [ ]:
# -----------------------------------------------------------
# step 16: select categorical columns and encoding method (ui)
# -----------------------------------------------------------
# this step loads the dataset containing rolling features:
#
#   /content/drive/MyDrive/ml_project/df_rolling.pkl
#
# the ui performs the following:
#   • automatically detects all columns with dtype 'object'
#   • displays them in a multi-select widget
#   • allows the user to choose one or more categorical columns
#   • provides a dropdown to select the encoding method:
#         onehot → create binary dummy variables
#         mean   → replace categories with mean(target | category)
#
# the user selections are saved into:
#
#   /content/drive/MyDrive/ml_project/encode_method.txt
#
# file format:
#   line 1 = comma-separated selected column names
#   line 2 = encoding method ('onehot' or 'mean')
#
# input file:
#   df_rolling.pkl
#
# output file:
#   encode_method.txt
encode_categories_ui()


In [ ]:
# -----------------------------------------------------------
# step 17: encode selected categorical columns and create df_encoded.pkl
# -----------------------------------------------------------
# this step:
#   1. loads the dataset:
#        /content/drive/MyDrive/ml_project/df_rolling.pkl
#   2. loads encoding configuration:
#        /content/drive/MyDrive/ml_project/encode_method.txt
#      (line 1: selected columns, line 2: encoding method)
#   3. applies the selected encoding:
#
#        onehot:
#            • uses pandas.get_dummies()
#            • creates one binary column per category
#            • removes the original categorical column
#
#        mean:
#            • requires the dataframe to contain a 'target' column
#            • computes mean(target | category)
#            • replaces each category with its group mean
#
#   4. saves the fully encoded dataframe as:
#        /content/drive/MyDrive/ml_project/df_encoded.pkl
#
# input files:
#   df_rolling.pkl
#   encode_method.txt
#
# output file:
#   df_encoded.pkl
#
# the encoded dataframe is returned for preview.
encode_categories()


saved: /content/drive/MyDrive/ml_project/df_encoded.pkl
encoded categories: ['PO_ID', 'Supplier', 'Item_Category', 'Order_Status']
method used: onehot


,Order_Date,Delivery_Date,Quantity,Unit_Price,year,month,day,dayofweek,Order_Date_lag_5,PO_ID_PO-00001,...,Supplier_Gamma_Co,Item_Category_Electronics,Item_Category_MRO,Item_Category_Office Supplies,Item_Category_Packaging,Item_Category_Raw Materials,Order_Status_Cancelled,Order_Status_Delivered,Order_Status_Partially Delivered,Order_Status_Pending
558,2022-01-01,2022-01-07,1042,36.74,2022,1,1,5,NaT,0,...,0,0,0,0,0,1,0,1,0,0
179,2022-01-01,2022-01-12,552,21.52,2022,1,1,5,NaT,0,...,0,1,0,0,0,0,0,1,0,0
521,2022-01-02,2022-01-06,85,22.54,2022,1,2,6,NaT,0,...,0,0,0,0,1,0,0,1,0,0
302,2022-01-03,2022-01-20,172,63.61,2022,1,3,0,NaT,0,...,0,0,0,0,1,0,0,1,0,0
420,2022-01-04,2022-01-06,1004,50.70,2022,1,4,1,NaT,0,...,0,1,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
663,2023-12-25,2024-01-12,1732,15.37,2023,12,25,0,2023-12-20,0,...,0,1,0,0,0,0,0,1,0,0
232,2023-12-26,2024-01-04,1546,43.47,2023,12,26,1,2023-12-20,0,...,0,0,1,0,0,0,0,1,0,0
519,2023-12-27,2023-12-28,952,74.72,2023,12,27,2,2023-12-22,0,...,0,0,0,0,0,1,0,1,0,0
210,2024-01-01,NaN,1233,86.62,2024,1,1,0,2023-12-23,0,...,1,0,0,1,0,0,0,1,0,0


In [ ]:
# -----------------------------------------------------------
# step 18: select the target column (ui)
# -----------------------------------------------------------
# this step loads the fully encoded dataset:
#
#   /content/drive/MyDrive/ml_project/df_encoded.pkl
#
# the ui displays all available columns and allows the user to pick
# exactly one column to serve as the target variable (y) for modeling.
#
# the selected column name is saved into:
#
#   /content/drive/MyDrive/ml_project/target_method.txt
#
# file format:
#   line 1 = name of the target column
#
# input file:
#   df_encoded.pkl
#
# output file:
#   target_method.txt
set_target_ui()


In [ ]:
# -----------------------------------------------------------
# step 19: separate features (X) and target (y) and save them
# -----------------------------------------------------------
# this step:
#   1. loads the encoded dataset:
#        /content/drive/MyDrive/ml_project/df_encoded.pkl
#   2. loads the selected target column name from:
#        /content/drive/MyDrive/ml_project/target_method.txt
#   3. splits the dataframe into:
#        X → all columns except the target
#        y → only the selected target column
#   4. saves both objects:
#        /content/drive/MyDrive/ml_project/X.pkl
#        /content/drive/MyDrive/ml_project/y.pkl
#
# input files:
#   df_encoded.pkl
#   target_method.txt
#
# output files:
#   X.pkl
#   y.pkl
#
# the function returns X and y for further processing.
X, y = set_target()


saved: /content/drive/MyDrive/ml_project/X.pkl
saved: /content/drive/MyDrive/ml_project/y.pkl
X and y successfully created


(    Delivery_Date  Quantity  Unit_Price  year  month  day  dayofweek  \
 558    2022-01-07      1042       36.74  2022      1    1          5   
 179    2022-01-12       552       21.52  2022      1    1          5   
 521    2022-01-06        85       22.54  2022      1    2          6   
 302    2022-01-20       172       63.61  2022      1    3          0   
 420    2022-01-06      1004       50.70  2022      1    4          1   
 ..            ...       ...         ...   ...    ...  ...        ...   
 663    2024-01-12      1732       15.37  2023     12   25          0   
 232    2024-01-04      1546       43.47  2023     12   26          1   
 519    2023-12-28       952       74.72  2023     12   27          2   
 210           NaN      1233       86.62  2024      1    1          0   
 649    2024-01-04      1327       59.57  2024      1    1          0   
 
     Order_Date_lag_5  PO_ID_PO-00001  PO_ID_PO-00002  ...  Supplier_Gamma_Co  \
 558              NaT               0    

In [ ]:
# -----------------------------------------------------------
# step 20: select numerical columns to scale (ui)
# -----------------------------------------------------------
# this step loads the feature dataset:
#
#   /content/drive/MyDrive/ml_project/X.pkl
#
# the ui automatically detects all numerical columns in X
# (only int64 and float64 dtypes are included).
#
# the user manually selects which of these numerical columns
# should be standardized using StandardScaler.
#
# the selected column names are saved into:
#
#   /content/drive/MyDrive/ml_project/scale_method.txt
#
# file format:
#   line 1 = comma-separated list of column names to scale
#
# input file:
#   X.pkl
#
# output file:
#   scale_method.txt
scale_data_ui()


In [ ]:
# -----------------------------------------------------------
# step 21: apply scaling and create X_scaled.pkl
# -----------------------------------------------------------
# this step:
#   1. loads the feature dataset:
#        /content/drive/MyDrive/ml_project/X.pkl
#   2. loads the user’s scaling configuration:
#        /content/drive/MyDrive/ml_project/scale_method.txt
#   3. extracts the selected numerical columns
#   4. applies StandardScaler ONLY to the selected columns
#      (other columns remain unchanged)
#   5. creates a new dataframe with identical structure:
#        X_scaled
#   6. saves the scaled dataset as:
#        /content/drive/MyDrive/ml_project/X_scaled.pkl
#
# input files:
#   X.pkl
#   scale_method.txt
#
# output file:
#   X_scaled.pkl
#
# the function returns X_scaled for further steps.
X_scaled = scale_data()
X_scaled.head()


saved: /content/drive/MyDrive/ml_project/X_scaled.pkl
scaled columns: ['Quantity', 'Unit_Price']


,Delivery_Date,Quantity,Unit_Price,year,month,day,dayofweek,Order_Date_lag_5,PO_ID_PO-00001,PO_ID_PO-00002,...,Supplier_Gamma_Co,Item_Category_Electronics,Item_Category_MRO,Item_Category_Office Supplies,Item_Category_Packaging,Item_Category_Raw Materials,Order_Status_Cancelled,Order_Status_Delivered,Order_Status_Partially Delivered,Order_Status_Pending
558,2022-01-07,-0.081338,-0.767142,2022,1,1,5,NaT,0,0,...,0,0,0,0,0,1,0,1,0,0
179,2022-01-12,-0.838179,-1.309103,2022,1,1,5,NaT,0,0,...,0,1,0,0,0,0,0,1,0,0
521,2022-01-06,-1.559495,-1.272782,2022,1,2,6,NaT,0,0,...,0,0,0,0,1,0,0,1,0,0
302,2022-01-20,-1.425117,0.189657,2022,1,3,0,NaT,0,0,...,0,0,0,0,1,0,0,1,0,0
420,2022-01-06,-0.140031,-0.270048,2022,1,4,1,NaT,0,0,...,0,1,0,0,0,0,0,1,0,0


In [ ]:
# -----------------------------------------------------------
# step 22: select the test set proportion (ui)
# -----------------------------------------------------------
# this step displays a simple ui that asks the user to enter the
# test size proportion, for example:
#       0.2 → 20% of the data used for testing
#
# the chosen value is saved into:
#
#   /content/drive/MyDrive/ml_project/test_size.txt
#
# file format:
#   line 1 = floating-point number (e.g., "0.2")
#
# input:  (no input files required)
#
# output file:
#   test_size.txt
split_time_ui()


In [ ]:
# -----------------------------------------------------------
# step 23: split the dataset chronologically into train and test
# -----------------------------------------------------------
# this step:
#   1. loads the feature dataset:
#        /content/drive/MyDrive/ml_project/X_scaled.pkl
#      (if X_scaled.pkl does not exist, it falls back to X.pkl)
#
#   2. loads the target dataset:
#        /content/drive/MyDrive/ml_project/y.pkl
#
#   3. loads the chosen test size:
#        /content/drive/MyDrive/ml_project/test_size.txt
#
#   4. computes the split point chronologically:
#        split_point = int(len(X) * (1 - test_size))
#
#   5. splits both X and y in time order (no shuffling):
#        X_train = first rows
#        X_test  = last rows
#        y_train = first rows
#        y_test  = last rows
#
#   6. saves the results into four separate files:
#        /content/drive/MyDrive/ml_project/X_train.pkl
#        /content/drive/MyDrive/ml_project/X_test.pkl
#        /content/drive/MyDrive/ml_project/y_train.pkl
#        /content/drive/MyDrive/ml_project/y_test.pkl
#
# input files:
#   X_scaled.pkl  (or X.pkl)
#   y.pkl
#   test_size.txt
#
# output files:
#   X_train.pkl
#   X_test.pkl
#   y_train.pkl
#   y_test.pkl
#
# the function returns the four datasets for verification.
x_train, x_test, y_train, y_test = split_time()
x_train.head()


saved: /content/drive/MyDrive/ml_project/X_train.pkl
saved: /content/drive/MyDrive/ml_project/X_test.pkl
saved: /content/drive/MyDrive/ml_project/y_train.pkl
saved: /content/drive/MyDrive/ml_project/y_test.pkl
split completed


,Delivery_Date,Quantity,Unit_Price,year,month,day,dayofweek,Order_Date_lag_5,PO_ID_PO-00001,PO_ID_PO-00002,...,Supplier_Gamma_Co,Item_Category_Electronics,Item_Category_MRO,Item_Category_Office Supplies,Item_Category_Packaging,Item_Category_Raw Materials,Order_Status_Cancelled,Order_Status_Delivered,Order_Status_Partially Delivered,Order_Status_Pending
558,2022-01-07,-0.081338,-0.767142,2022,1,1,5,NaT,0,0,...,0,0,0,0,0,1,0,1,0,0
179,2022-01-12,-0.838179,-1.309103,2022,1,1,5,NaT,0,0,...,0,1,0,0,0,0,0,1,0,0
521,2022-01-06,-1.559495,-1.272782,2022,1,2,6,NaT,0,0,...,0,0,0,0,1,0,0,1,0,0
302,2022-01-20,-1.425117,0.189657,2022,1,3,0,NaT,0,0,...,0,0,0,0,1,0,0,1,0,0
420,2022-01-06,-0.140031,-0.270048,2022,1,4,1,NaT,0,0,...,0,1,0,0,0,0,0,1,0,0


In [ ]:
# -----------------------------------------------------------
# step 24: select traditional forecasting model (ui)
# -----------------------------------------------------------
# this step:
#   1. shows an interactive ui where the user selects one of the
#      traditional forecasting models:
#         - naive
#         - moving_average
#         - simple_exponential_smoothing
#         - arima
#
#   2. the ui also lets the user enter model parameters:
#         moving_average: one integer, example: 3
#         arima: three integers separated by commas, example: 1,1,1
#         naive and ses do not require parameters
#
#   3. the selected model and parameters are saved to:
#         /content/drive/MyDrive/ml_project/traditional_method.txt
#
# this step does not train any model and does not produce forecasts.
# it only saves configuration that will be used later.
#
# input files:
#   none
#
# output files:
#   traditional_method.txt
#
create_traditional_model_ui()


In [ ]:
# -----------------------------------------------------------
# step 25: select machine learning model (ui)
# -----------------------------------------------------------
# this step:
#   1. does not load any dataset, because choosing the ml model
#      and the parameter selection mode does not depend on x or y
#
#   2. shows an interactive ui where the user selects one of the
#      machine learning models:
#         - linear_regression
#         - ridge
#         - lasso
#         - random_forest_regressor
#         - svr
#         - xgboost_regressor
#         - mlp_regressor
#
#   3. the ui also asks the user to choose the parameter selection mode:
#         - manual         (user enters parameters directly)
#         - grid_search    (user enters a parameter grid)
#         - random_search  (user enters parameter ranges)
#
#   4. depending on the selected mode, only one parameter input field
#      becomes visible:
#         manual: raw parameter string, e.g. alpha=1,max_depth=5
#         grid_search: parameter grid, e.g. alpha=[0.1,1,10]
#         random_search: numeric ranges, e.g. n_estimators=100-500
#
#   5. after confirmation, three lines are saved into:
#         /content/drive/MyDrive/ml_project/ml_method.txt
#         line 1: selected model name
#         line 2: selected mode
#         line 3: parameter string (manual, grid, or random format)
#
#   6. this step only saves the configuration. no model is created,
#      trained, or evaluated here.
#
# input files:
#   none
#
# output files:
#   ml_method.txt
#
create_ml_model_ui()



In [ ]:
# -----------------------------------------------------------
# step 26: train the machine learning model
# -----------------------------------------------------------
# this step:
#   1. loads the model configuration saved earlier in:
#        /content/drive/MyDrive/ml_project/ml_method.txt
#      the file contains:
#        - selected model name
#        - selected parameter mode (manual, grid_search, random_search)
#        - parameter string entered by the user
#
#   2. loads the training datasets created during the time split:
#        /content/drive/MyDrive/ml_project/X_train.pkl
#        /content/drive/MyDrive/ml_project/y_train.pkl
#
#   3. creates the base model according to the selected name:
#        - linear_regression
#        - ridge
#        - lasso
#        - random_forest_regressor
#        - svr
#        - xgboost_regressor
#        - mlp_regressor
#
#   4. applies the chosen parameter selection mode:
#        manual:
#            parameters are applied directly to the model and the model is trained
#        grid_search:
#            gridsearchcv is run with the provided parameter grid
#            best estimator becomes the final trained model
#        random_search:
#            randomizedsearchcv is run with the provided parameter ranges
#            best estimator becomes the final trained model
#
#   5. the final trained model is saved into:
#        /content/drive/MyDrive/ml_project/ml_trained.pkl
#
# input files:
#   ml_method.txt
#   X_train.pkl
#   y_train.pkl
#
# output files:
#   ml_trained.pkl
#
ml_model = train_ml()
ml_model


In [ ]:
# -----------------------------------------------------------
# step 27: generate ml predictions on the test dataset
# -----------------------------------------------------------
# this step:
#   1. loads the trained ml model:
#        /content/drive/MyDrive/ml_project/ml_trained.pkl
#
#   2. loads the test feature dataset:
#        /content/drive/MyDrive/ml_project/X_test.pkl
#
#   3. applies the trained model to generate predictions:
#        y_pred = model.predict(X_test)
#
#   4. converts the predictions to a numpy array and saves them to:
#        /content/drive/MyDrive/ml_project/y_pred_ml.pkl
#
#   5. this step does not evaluate forecast accuracy.
#      it only creates raw ml predictions that will be used later.
#
# input files:
#   ml_trained.pkl
#   X_test.pkl
#
# output files:
#   y_pred_ml.pkl
#
y_pred_ml = predict_ml()


In [ ]:
# -----------------------------------------------------------
# step 28: train the traditional forecasting model
# -----------------------------------------------------------
# this step:
#   1. loads the traditional model configuration saved earlier in:
#        /content/drive/MyDrive/ml_project/traditional_method.txt
#      the file contains:
#        - selected traditional model name
#        - selected parameters (if any)
#
#   2. loads the training dataset created during the time split:
#        /content/drive/MyDrive/ml_project/y_train.pkl
#
#   3. trains only the models that require fitting:
#        - simple_exponential_smoothing
#        - arima
#      naive and moving_average do not require training
#
#   4. saves the trained model into:
#        /content/drive/MyDrive/ml_project/traditional_trained.pkl
#
#   5. saves additional information about the training process into:
#        /content/drive/MyDrive/ml_project/traditional_info.txt
#
# input files:
#   traditional_method.txt
#   y_train.pkl
#
# output files:
#   traditional_trained.pkl
#   traditional_info.txt
#
traditional_model = train_traditional()
traditional_model